In [1]:
# import snowflake
# from snowflake.snowpark.context import get_active_session
# session = get_active_session()

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# ----- 1) Imports & Reproducibility -----
import os
import math
import random
import numpy as np
import pandas as pd
import re

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

In [29]:
Water_Quality_df = pd.read_csv("data/water_quality_training_dataset.csv")
landsat_train_features = pd.read_csv("data/landsat/landsat_features_training_mvdb.csv")
Terraclimate_df = pd.read_csv("data/terraclimate_features_training_combined.csv")

#Clean Up the Data
#Capitalize Col Names
def capitalize_words_keep_spacing(col):
    # Split on space or underscore but keep the separators
    parts = re.split(r'([ _])', col)
    # Capitalize text parts, keep separators unchanged
    return ''.join(p.title() if p not in [' ', '_'] else p for p in parts)
Terraclimate_df.columns = [capitalize_words_keep_spacing(c) for c in Terraclimate_df.columns]
landsat_train_features.columns = [capitalize_words_keep_spacing(c) for c in landsat_train_features.columns]
Water_Quality_df.columns = [capitalize_words_keep_spacing(c) for c in Water_Quality_df.columns]

def combine_two_datasets(dataset1,dataset2,dataset3):
    '''
    Returns a  vertically concatenated dataset.
    Attributes:
    dataset1 - Dataset 1 to be combined 
    dataset2 - Dataset 2 to be combined
    '''
    
    data = pd.concat([dataset1,dataset2,dataset3], axis=1)
    data = data.loc[:, ~data.columns.duplicated()]
    return data

wq_data = combine_two_datasets(Water_Quality_df, landsat_train_features, Terraclimate_df)

#Data type corrections 
wq_data['Sample Date'] = pd.to_datetime(wq_data['Sample Date'],  format='%d-%m-%Y')


def convert_text_cols_to_float(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        if (pd.api.types.is_object_dtype(out[c]) 
            or pd.api.types.is_string_dtype(out[c]) 
            or pd.api.types.is_categorical_dtype(out[c])):
            s = out[c].astype(str).str.replace('\u00A0', ' ', regex=False).str.strip()
            s = s.str.replace('\u2212', '-', regex=False)                 # unicode minus
            s = s.str.replace(r'^\(\s*(.*)\s*\)$', r'-\1', regex=True)    # (123) -> -123
            s = s.str.replace(r'^\s*([+]?\s*[\d, .]+)\s*-$', r'-\1', regex=True)  # 123- -> -123
            s = s.str.replace(r'(?<=\d),(?=\d{3}\b)', '', regex=True)      # thousands comma
            out[c] = pd.to_numeric(s, errors='coerce')                     # float64 by default with NaNs
    return out

wq_data = convert_text_cols_to_float(wq_data)

#ullify all negative observations
for column in wq_data.columns:
    if column != "Sample Date": wq_data[wq_data[column] < -9000][column] = np.nan 

wq_data['Month_cosine'] = np.cos((wq_data['Sample Date'].dt.month + (wq_data['Sample Date'].dt.day/31))* np.pi / 6)

#wq_data = wq_data.drop(columns=['Qa_Radsat', 'Cloud_Qa', 'Sample Date',  'Atmos_Opacity', 'Emsd', 'Lwir', 'Unnamed: 0'])
#, 'Atmos opacity, 'Emsd', 'Lwir''
feature_cols = wq_data.columns.tolist()

#print(wq_data.columns)
feature_cols.remove('Latitude')
feature_cols.remove('Longitude')
feature_cols.remove('Month_cosine')
feature_cols.remove('Total Alkalinity')
feature_cols.remove('Electrical Conductance')
feature_cols.remove('Dissolved Reactive Phosphorus')

wq_data = wq_data.dropna(subset=feature_cols)

wq_data = wq_data.drop(columns='Sample Date')


In [31]:
Water_Quality_df_v = pd.read_csv("data/submission_template.csv")
landsat_train_features_v = pd.read_csv("data/landsat/landsat_features_validation_mvdb.csv")
Terraclimate_df_v = pd.read_csv("data/terraclimate_features_validation_combined.csv")

#Clean Up the Data
Terraclimate_df_v.columns = [capitalize_words_keep_spacing(c) for c in Terraclimate_df_v.columns]
landsat_train_features_v.columns = [capitalize_words_keep_spacing(c) for c in landsat_train_features_v.columns]
Water_Quality_df_v.columns = [capitalize_words_keep_spacing(c) for c in Water_Quality_df_v.columns]

wq_data_v = combine_two_datasets(Water_Quality_df_v, landsat_train_features_v, Terraclimate_df_v)

#Data type corrections 
wq_data_v['Sample Date'] = pd.to_datetime(wq_data_v['Sample Date'],  format='%d-%m-%Y')
wq_data_v = convert_text_cols_to_float(wq_data_v)

#wq_data_v = wq_data_v.drop(columns=['Qa_Radsat', 'Cloud_Qa', 'Atmos_Opacity', 'Emsd', 'Lwir', 'Unnamed: 0'])
#ullify all negative observations
for column in wq_data_v.columns:
    if column != "Sample Date": wq_data_v[wq_data_v[column] < -9000][column] = np.nan 

wq_data_v['Month_cosine'] = np.cos((wq_data_v['Sample Date'].dt.month + (wq_data_v['Sample Date'].dt.day/31))* np.pi / 6)

wq_data_final = wq_data_v[['Latitude', 'Longitude', 'Sample Date', 'Month_cosine']]


wq_data_v = wq_data_v.dropna(subset=feature_cols)

wq_data_v = wq_data_v.drop(columns='Sample Date')

wq_data_v = wq_data_v[sorted(wq_data_v.columns)]

print(wq_data_final.columns)

Index(['Latitude', 'Longitude', 'Sample Date', 'Month_cosine'], dtype='object')


In [32]:
# For reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)



Device: cpu


In [33]:
#number of cv groups
cv_groups = 8

#split over longitude
wq_data['cv_group'] = pd.qcut(wq_data['Latitude'], q=cv_groups, labels=False)

# Specify the number of folds
lat_sep_kf = GroupKFold(n_splits=cv_groups - 1)

In [ ]:
ava_cols = wq_data.columns.tolist()

#print(wq_data.columns)
ava_cols.remove('Latitude')
ava_cols.remove('Longitude')
ava_cols.remove('Month_cosine')
ava_cols.remove('Total Alkalinity')
ava_cols.remove('Electrical Conductance')
ava_cols.remove('Dissolved Reactive Phosphorus')

'Nvdi', 'Savi', 'Ppt', 'Q', 'Bsi', 'Ndbi', 'Tmax', 'Vpd', 'Pet', 'Aet', 'Soil'

print(ava_cols)

ValueError: list.remove(x): x not in list

In [35]:
def normalize_features(feat_means, feat_stds, frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out[feature_cols] = (out[feature_cols] - feat_means) / feat_stds
    return out


# ----- 7) PyTorch Dataset -----
class TabularDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, key_cols, feature_cols, target_cols, task_type="regression"):
        self.keys = frame[key_cols].reset_index(drop=True)
        X = frame[feature_cols].to_numpy(dtype=np.float32)
        self.X = torch.from_numpy(X)
        y = frame[target_cols].to_numpy(dtype=np.float32)
        self.y = torch.from_numpy(y)
        self.task_type = task_type

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        # Return keys (as dict-like), features, and targets
        return self.keys.iloc[idx].to_dict(), self.X[idx], self.y[idx]
    
# ----- 9) Model: a simple MLP -----
class MLP(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, hidden_dims=(128, 64), dropout=0.1, task_type="regression"):
        super().__init__()
        layers = []
        last = in_dim
        for h in hidden_dims:
            layers += [nn.Linear(last, h), nn.ReLU(), nn.Dropout(dropout)]
            last = h
        layers.append(nn.Linear(last, out_dim))
        self.net = nn.Sequential(*layers)
        self.task_type = task_type

    def forward(self, x):
        logits_or_outputs = self.net(x)
        # For regression, return raw outputs.
        # For multilabel classification: during training we keep raw logits (BCEWithLogitsLoss expects logits).
        return logits_or_outputs


In [36]:
# ----- 11) Metrics -----
@torch.no_grad()
def evaluate(model, loader, task_type="regression", threshold=0.5):

    preds_list = []
    targets_list = []


    model.eval()
    total_loss = 0.0
    n_examples = 0

    # For regression: track MAE and RMSE
    mae_sum = torch.zeros(len(target_cols), device=DEVICE)
    mse_sum = torch.zeros(len(target_cols), device=DEVICE)

    # For multilabel: track accuracy (micro)
    correct = 0
    total   = 0

    for _, X, y in loader:
        X = X.to(DEVICE)
        y = y.to(DEVICE)
        out = model(X)

        preds_list.append(out.detach().cpu())
        targets_list.append(y.detach().cpu())

        loss = criterion(out, y)

        if torch.isnan(loss) or torch.isinf(loss):
            print("Bad loss detected!")
            print("out min/max:", out.min().item(), out.max().item())
            print("y min/max:", y.min().item(), y.max().item())
            print("loss:", loss)
            return {"loss": float("nan")}

        bs = y.size(0)
        total_loss += loss.item() * bs
        n_examples += bs

        if task_type == "regression":
            err = torch.abs(out - y)
            mae_sum += err.sum(dim=0)
            mse_sum += ((out - y) ** 2).sum(dim=0)
        else:
            # multilabel: apply sigmoid then threshold
            probs = torch.sigmoid(out)
            preds = (probs >= threshold).float()
            correct += (preds == y).sum().item()
            total   += y.numel()

    avg_loss = total_loss / max(n_examples, 1)

    if task_type == "regression":

        preds = torch.cat(preds_list, dim=0)
        targets = torch.cat(targets_list, dim=0)

        mae = (mae_sum / n_examples).detach().cpu().numpy()
        rmse = torch.sqrt(mse_sum / n_examples).detach().cpu().numpy()

        # Compute R2 per target
        ss_res = torch.sum((preds - targets) ** 2, dim=0)
        ss_tot = torch.sum((targets - targets.mean(dim=0)) ** 2, dim=0)

        r2 = 1 - ss_res / torch.clamp(ss_tot, min=1e-12)
        r2 = r2.numpy()

        metrics = {
            "loss": avg_loss,
            "MAE_per_target": {t: float(mae[i]) for i, t in enumerate(target_cols)},
            "RMSE_per_target": {t: float(rmse[i]) for i, t in enumerate(target_cols)},
            "R2_per_target": {t: float(r2[i]) for i, t in enumerate(target_cols)},
        }
    else:
        accuracy = correct / max(total, 1)
        metrics = {"loss": avg_loss, "micro_accuracy": accuracy}

    return metrics

In [ ]:
def metrics_to_dataframe(metrics):
    df = pd.DataFrame({
        "MAE": metrics["MAE_per_target"],
        "RMSE": metrics["RMSE_per_target"],
        "R2": metrics["R2_per_target"]
    })

    # Add overall loss as a separate column (same value for all rows)
    df["Loss"] = metrics["loss"]

    df = df.T

    return df



In [45]:
all_metrics = []

for i in ava_cols:
   
   if i in ['Nvdi', 'Savi', 'Ppt', 'Q', 'Bsi', 'Ndbi', 'Tmax', 'Vpd', 'Pet', 'Aet', 'Soil', 'cv_group']:
      continue

   print(i)
   
   wq_data_mini = wq_data[['Latitude', 'Longitude', 'Month_cosine', 'Total Alkalinity',
      'Electrical Conductance', 'Dissolved Reactive Phosphorus', 'Nvdi', 'Savi', 'Ppt', 'Q', 'Bsi', 'Ndbi', 'Tmax', 'Vpd', 'Pet', 'Aet', 'Soil', 'cv_group', i]]
   wq_data_v_mini = wq_data_v[['Latitude', 'Longitude', 'Month_cosine', 'Total Alkalinity',
      'Electrical Conductance', 'Dissolved Reactive Phosphorus', 'Nvdi', 'Savi', 'Ppt', 'Q', 'Bsi', 'Ndbi', 'Tmax', 'Vpd', 'Pet', 'Aet', 'Soil', i]]
   
   #test train based on location
   wq_data_test = wq_data_mini[wq_data_mini['cv_group'] == 7]
   wq_data_train = wq_data_mini[wq_data_mini['cv_group'] != 7]

   wq_data_test = wq_data_test.drop(columns='cv_group')
   wq_data_train = wq_data_train.drop(columns='cv_group')

   wq_data_train = wq_data_train[sorted(wq_data_train.columns)]
   wq_data_test = wq_data_test[sorted(wq_data_test.columns)]

   # ----- 2) Configurable column names & task type -----
   # Adjust these lists for your real dataset
   key_cols     = ["Latitude", "Longitude", "Month_cosine"]  # 3 key columns (IDs, timestamps, etc.)
   feature_cols = wq_data_mini.columns.tolist()
   feature_cols.remove('Latitude')
   feature_cols.remove('Longitude')
   feature_cols.remove('Month_cosine')
   feature_cols.remove('Total Alkalinity')
   feature_cols.remove('Electrical Conductance')
   feature_cols.remove('Dissolved Reactive Phosphorus')
   feature_cols.remove('cv_group')
   #feature_cols = [f"f{i}" for i in range(1, 21)]  # 20 feature columns
   target_cols = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']
   #target_cols  = ["y1", "y2", "y3"]  # 3 target columns

   # Choose the task: "regression" (default) or "multilabel"
   #  - regression:  continuous y1,y2,y3  -> MSELoss
   #  - multilabel:  binary y1,y2,y3 in {0,1} -> BCEWithLogitsLoss
   task_type = "regression"

   # ----- 6) Compute normalization stats on TRAIN ONLY -----
   # Standardize features to zero-mean, unit-std
   feat_means = wq_data_train[feature_cols].mean()
   feat_stds  = wq_data_train[feature_cols].std().replace(0, 1.0)  # avoid division by zero

   df_train_n = normalize_features(feat_means, feat_stds, wq_data_train)
   df_val_n   = normalize_features(feat_means, feat_stds, wq_data_test)
   df_test_n  = normalize_features(feat_means, feat_stds, wq_data_v_mini)

   train_ds = TabularDataset(df_train_n, key_cols, feature_cols, target_cols, task_type=task_type)
   val_ds   = TabularDataset(df_val_n,   key_cols, feature_cols, target_cols, task_type=task_type)
   test_ds  = TabularDataset(df_test_n,  key_cols, feature_cols, target_cols, task_type=task_type)

   # ----- 8) DataLoaders -----
   BATCH_SIZE = 128
   train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=False)
   val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
   test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

   model = MLP(in_dim=len(feature_cols), out_dim=len(target_cols), hidden_dims=(128, 64), dropout=0.15, task_type=task_type).to(DEVICE)

   # ----- 10) Loss and Optimizer -----
   if task_type == "regression":
      criterion = nn.MSELoss()
   else:  # "multilabel"
      criterion = nn.BCEWithLogitsLoss()

   optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

   # ----- 12) Training Loop -----
   EPOCHS = 50
   best_val = math.inf
   patience = 10
   wait = 0
   best_state = None



   for epoch in range(1, EPOCHS + 1):
      model.train()
      running_loss = 0.0
      n_seen = 0

      for _, X, y in train_loader:
         X = X.to(DEVICE)
         y = y.to(DEVICE)

         optimizer.zero_grad()
         out = model(X)
         
         loss = criterion(out, y)
         loss.backward()
         optimizer.step()

         running_loss += loss.item() * X.size(0)
         n_seen += X.size(0)

      train_loss = running_loss / max(n_seen, 1)
      val_metrics = evaluate(model, val_loader, task_type=task_type)
      val_loss = val_metrics["loss"]

      #print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Metrics: {val_metrics}")

      # Simple early stopping on validation loss
      if val_loss < best_val - 1e-6:
         best_val = val_loss
         best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
         wait = 0
      else:
         wait += 1
         if wait >= patience:
               #print("Early stopping.")
               break

   # Load best model (if available)
   if best_state is not None:
      model.load_state_dict(best_state)

   test_metrics = evaluate(model, val_loader, task_type=task_type)
   print("TEST:", test_metrics)

   df_metrics = metrics_to_dataframe(test_metrics)

   df_metrics = df_metrics.rename(columns={'Total Alkalinity': i})

   all_metrics.append(df_metrics)


Emis
TEST: {'loss': 33996.644234572785, 'MAE_per_target': {'Total Alkalinity': 56.3114128112793, 'Electrical Conductance': 244.26446533203125, 'Dissolved Reactive Phosphorus': 40.66965866088867}, 'RMSE_per_target': {'Total Alkalinity': 69.85079193115234, 'Electrical Conductance': 307.31622314453125, 'Dissolved Reactive Phosphorus': 51.64823913574219}, 'R2_per_target': {'Total Alkalinity': -0.06564545631408691, 'Electrical Conductance': 0.0171201229095459, 'Dissolved Reactive Phosphorus': 0.09117388725280762}}
Atran
TEST: {'loss': 32463.353709877938, 'MAE_per_target': {'Total Alkalinity': 53.8586311340332, 'Electrical Conductance': 234.04298400878906, 'Dissolved Reactive Phosphorus': 40.2900276184082}, 'RMSE_per_target': {'Total Alkalinity': 68.79902648925781, 'Electrical Conductance': 299.9743957519531, 'Dissolved Reactive Phosphorus': 51.69264602661133}, 'R2_per_target': {'Total Alkalinity': -0.03379535675048828, 'Electrical Conductance': 0.06352156400680542, 'Dissolved Reactive Phosp

In [46]:
all_metrics_copy = all_metrics

In [47]:
print(all_metrics_copy)

[                                      MAE        RMSE        R2          Loss
Total Alkalinity                56.311413   69.850792 -0.065645  33996.644235
Electrical Conductance         244.264465  307.316223  0.017120  33996.644235
Dissolved Reactive Phosphorus   40.669659   51.648239  0.091174  33996.644235,                                       MAE        RMSE        R2         Loss
Total Alkalinity                53.858631   68.799026 -0.033795  32463.35371
Electrical Conductance         234.042984  299.974396  0.063522  32463.35371
Dissolved Reactive Phosphorus   40.290028   51.692646  0.089610  32463.35371,                                       MAE        RMSE        R2          Loss
Total Alkalinity                56.915478   69.636574 -0.059119  34035.215614
Electrical Conductance         244.165771  307.583740  0.015408  34035.215614
Dissolved Reactive Phosphorus   38.695168   51.464897  0.097615  34035.215614,                                       MAE        RMSE        R2 

In [19]:
# ----- 14) Inference: get predictions + keys side-by-side -----
@torch.no_grad()
def predict_with_keys(model, loader, task_type="regression", threshold=0.5):
    model.eval()
    all_rows = []
    for keys, X, _ in loader:
        X = X.to(DEVICE)
        logits_or_outputs = model(X).cpu().numpy()

        if task_type == "regression":
            preds = logits_or_outputs  # raw outputs
        else:
            probs = 1 / (1 + np.exp(-logits_or_outputs))  # sigmoid
            preds = (probs >= threshold).astype(np.float32)

        # For each batch row, combine keys + predictions into a dict
        for i in range(len(keys["Latitude"])):
            row = {k: keys[k][i] for k in keys.keys()}
            for j, t in enumerate(target_cols):
                row[f"pred_{t}"] = float(preds[i, j])
            all_rows.append(row)
    return pd.DataFrame(all_rows)

pred_df = predict_with_keys(model, test_loader, task_type=task_type)
pred_df.head(5)

,Latitude,Longitude,Month_cosine,pred_Total Alkalinity,pred_Electrical Conductance,pred_Dissolved Reactive Phosphorus
0,"tensor(-32.0433, dtype=torch.float64)","tensor(27.8228, dtype=torch.float64)","tensor(0.0169, dtype=torch.float64)",70.978340,285.681274,26.259087
1,"tensor(-33.3292, dtype=torch.float64)","tensor(26.0775, dtype=torch.float64)","tensor(0.2670, dtype=torch.float64)",86.484032,389.038147,29.438313
2,"tensor(-32.9916, dtype=torch.float64)","tensor(27.6400, dtype=torch.float64)","tensor(-0.9190, dtype=torch.float64)",75.312836,355.829956,26.718395
3,"tensor(-34.0964, dtype=torch.float64)","tensor(24.4392, dtype=torch.float64)","tensor(0.3944, dtype=torch.float64)",70.836098,324.822998,27.223091
4,"tensor(-32.0006, dtype=torch.float64)","tensor(28.5817, dtype=torch.float64)","tensor(0.5146, dtype=torch.float64)",104.095848,392.558594,41.135456


In [20]:
pred_df['Latitude'] = pred_df['Latitude'].apply(lambda x: x.item())
pred_df['Longitude'] = pred_df['Longitude'].apply(lambda x: x.item())
pred_df['Month_cosine'] = pred_df['Month_cosine'].apply(lambda x: x.item())

In [21]:
# Reverse cosine encoding
def reverse_month_cosine(cos_val):
    # Ensure value is in valid range for arccos
    cos_val = np.clip(cos_val, -1, 1)
    
    # Get angle in radians
    angle = np.arccos(cos_val)
    
    # Convert back to fractional month
    fractional_month = angle / (np.pi / 6)  # inverse of * np.pi/6
    
    # Approximate month and day
    month = int(fractional_month)  # integer month
    day = int(round((fractional_month - month) * 31))  # approximate day
    
    # Adjust for month starting at 1
    month = max(1, min(12, month))
    day = max(1, min(31, day))
    
    return month, day

# # Apply to DataFrame
# pred_df[['Approx_Month', 'Approx_Day']] = pred_df['Month_cosine'].apply(
#     lambda x: pd.Series(reverse_month_cosine(x))
# )

In [22]:
display(pred_df)

,Latitude,Longitude,Month_cosine,pred_Total Alkalinity,pred_Electrical Conductance,pred_Dissolved Reactive Phosphorus
0,-32.043333,27.822778,0.016889,70.978340,285.681274,26.259087
1,-33.329167,26.077500,0.266967,86.484032,389.038147,29.438313
2,-32.991639,27.640028,-0.918958,75.312836,355.829956,26.718395
3,-34.096389,24.439167,0.394356,70.836098,324.822998,27.223091
4,-32.000556,28.581667,0.514555,104.095848,392.558594,41.135456
...,...,...,...,...,...,...
195,-33.771111,25.386667,0.994869,100.868095,476.734741,34.619949
196,-33.185361,27.390750,0.067510,83.909988,394.903625,29.637768
197,-32.043333,27.822778,0.455495,86.201378,374.269623,29.881546
198,-33.001667,25.161389,0.790776,124.519836,478.200867,48.885880


In [ ]:
# # ----- 15) (Optional) Save artifacts -----
# os.makedirs("artifacts", exist_ok=True)
# torch.save({
#     "model_state_dict": model.state_dict(),
#     "feature_cols": feature_cols,
#     "target_cols": target_cols,
#     "feat_means": feat_means.to_dict(),
#     "feat_stds": feat_stds.to_dict(),
#     "task_type": task_type
# }, "artifacts/model.pt")

# pred_df.to_csv("artifacts/test_predictions_with_keys.csv", index=False)
# #print("Saved: artifacts/model.pt and artifacts/test_predictions_with_keys.csv")

In [23]:
condition = (pred_df['pred_Total Alkalinity'] > 20) & (pred_df['pred_Total Alkalinity'] < 200)

count1 = condition.sum()

print("Count using sum():", count1)


condition = (pred_df['pred_Electrical Conductance'] < 800)

count1 = condition.sum()

print("Count using sum():", count1)

condition = (pred_df['pred_Dissolved Reactive Phosphorus'] < 100)

count1 = condition.sum()

print("Count using sum():", count1)


Count using sum(): 199
Count using sum(): 199
Count using sum(): 200


In [135]:
pred_df.to_csv('data/output.csv', index=False)

In [48]:
import ast
import pandas as pd

raw_text = """
Emis
TEST: {'loss': 33996.644234572785, 'MAE_per_target': {'Total Alkalinity': 56.3114128112793, 'Electrical Conductance': 244.26446533203125, 'Dissolved Reactive Phosphorus': 40.66965866088867}, 'RMSE_per_target': {'Total Alkalinity': 69.85079193115234, 'Electrical Conductance': 307.31622314453125, 'Dissolved Reactive Phosphorus': 51.64823913574219}, 'R2_per_target': {'Total Alkalinity': -0.06564545631408691, 'Electrical Conductance': 0.0171201229095459, 'Dissolved Reactive Phosphorus': 0.09117388725280762}}
Atran
TEST: {'loss': 32463.353709877938, 'MAE_per_target': {'Total Alkalinity': 53.8586311340332, 'Electrical Conductance': 234.04298400878906, 'Dissolved Reactive Phosphorus': 40.2900276184082}, 'RMSE_per_target': {'Total Alkalinity': 68.79902648925781, 'Electrical Conductance': 299.9743957519531, 'Dissolved Reactive Phosphorus': 51.69264602661133}, 'R2_per_target': {'Total Alkalinity': -0.03379535675048828, 'Electrical Conductance': 0.06352156400680542, 'Dissolved Reactive Phosphorus': 0.08961021900177002}}
Swir16
TEST: {'loss': 34035.21561369801, 'MAE_per_target': {'Total Alkalinity': 56.91547775268555, 'Electrical Conductance': 244.165771484375, 'Dissolved Reactive Phosphorus': 38.695167541503906}, 'RMSE_per_target': {'Total Alkalinity': 69.6365737915039, 'Electrical Conductance': 307.583740234375, 'Dissolved Reactive Phosphorus': 51.46489715576172}, 'R2_per_target': {'Total Alkalinity': -0.05911898612976074, 'Electrical Conductance': 0.015408217906951904, 'Dissolved Reactive Phosphorus': 0.09761470556259155}}
Atmos_Opacity
TEST: {'loss': 33346.329205752714, 'MAE_per_target': {'Total Alkalinity': 54.75972366333008, 'Electrical Conductance': 238.63528442382812, 'Dissolved Reactive Phosphorus': 39.458744049072266}, 'RMSE_per_target': {'Total Alkalinity': 69.32300567626953, 'Electrical Conductance': 304.37408447265625, 'Dissolved Reactive Phosphorus': 50.88937759399414}, 'R2_per_target': {'Total Alkalinity': -0.04960227012634277, 'Electrical Conductance': 0.035849571228027344, 'Dissolved Reactive Phosphorus': 0.1176842451095581}}
Green
TEST: {'loss': 33954.91454283454, 'MAE_per_target': {'Total Alkalinity': 54.71660614013672, 'Electrical Conductance': 242.30885314941406, 'Dissolved Reactive Phosphorus': 41.00852966308594}, 'RMSE_per_target': {'Total Alkalinity': 69.90382385253906, 'Electrical Conductance': 307.2462463378906, 'Dissolved Reactive Phosphorus': 50.77348327636719}, 'R2_per_target': {'Total Alkalinity': -0.06726396083831787, 'Electrical Conductance': 0.017567753791809082, 'Dissolved Reactive Phosphorus': 0.12169843912124634}}
Drad
TEST: {'loss': 33940.715783227846, 'MAE_per_target': {'Total Alkalinity': 55.201210021972656, 'Electrical Conductance': 241.69540405273438, 'Dissolved Reactive Phosphorus': 39.30297088623047}, 'RMSE_per_target': {'Total Alkalinity': 69.18372344970703, 'Electrical Conductance': 307.27374267578125, 'Dissolved Reactive Phosphorus': 51.17221450805664}, 'R2_per_target': {'Total Alkalinity': -0.04538869857788086, 'Electrical Conductance': 0.017391622066497803, 'Dissolved Reactive Phosphorus': 0.10784941911697388}}
Red
TEST: {'loss': 33618.57635553232, 'MAE_per_target': {'Total Alkalinity': 55.3410758972168, 'Electrical Conductance': 241.37245178222656, 'Dissolved Reactive Phosphorus': 39.269317626953125}, 'RMSE_per_target': {'Total Alkalinity': 69.07093048095703, 'Electrical Conductance': 305.77227783203125, 'Dissolved Reactive Phosphorus': 50.87477111816406}, 'R2_per_target': {'Total Alkalinity': -0.04198324680328369, 'Electrical Conductance': 0.026971042156219482, 'Dissolved Reactive Phosphorus': 0.1181907057762146}}
Swir22
TEST: {'loss': 33975.77530232821, 'MAE_per_target': {'Total Alkalinity': 56.453269958496094, 'Electrical Conductance': 244.7124481201172, 'Dissolved Reactive Phosphorus': 40.23776626586914}, 'RMSE_per_target': {'Total Alkalinity': 69.46477508544922, 'Electrical Conductance': 307.36724853515625, 'Dissolved Reactive Phosphorus': 51.257835388183594}, 'R2_per_target': {'Total Alkalinity': -0.053899526596069336, 'Electrical Conductance': 0.016793906688690186, 'Dissolved Reactive Phosphorus': 0.10486149787902832}}
Emsd
TEST: {'loss': 34106.27448999774, 'MAE_per_target': {'Total Alkalinity': 54.943206787109375, 'Electrical Conductance': 242.96685791015625, 'Dissolved Reactive Phosphorus': 40.08555603027344}, 'RMSE_per_target': {'Total Alkalinity': 69.33123779296875, 'Electrical Conductance': 308.07476806640625, 'Dissolved Reactive Phosphorus': 51.009246826171875}, 'R2_per_target': {'Total Alkalinity': -0.04985165596008301, 'Electrical Conductance': 0.012262105941772461, 'Dissolved Reactive Phosphorus': 0.11352270841598511}}
Cdist
TEST: {'loss': 32616.734276107596, 'MAE_per_target': {'Total Alkalinity': 53.43575668334961, 'Electrical Conductance': 238.5951690673828, 'Dissolved Reactive Phosphorus': 39.163124084472656}, 'RMSE_per_target': {'Total Alkalinity': 67.94554901123047, 'Electrical Conductance': 301.2057189941406, 'Dissolved Reactive Phosphorus': 50.08712387084961}, 'R2_per_target': {'Total Alkalinity': -0.00830531120300293, 'Electrical Conductance': 0.055817604064941406, 'Dissolved Reactive Phosphorus': 0.14528387784957886}}
Nir08
TEST: {'loss': 33838.18662409584, 'MAE_per_target': {'Total Alkalinity': 53.80026626586914, 'Electrical Conductance': 239.8011474609375, 'Dissolved Reactive Phosphorus': 40.03615951538086}, 'RMSE_per_target': {'Total Alkalinity': 69.86927795410156, 'Electrical Conductance': 306.626220703125, 'Dissolved Reactive Phosphorus': 51.11954879760742}, 'R2_per_target': {'Total Alkalinity': -0.06620931625366211, 'Electrical Conductance': 0.021528899669647217, 'Dissolved Reactive Phosphorus': 0.10968494415283203}}
Blue
TEST: {'loss': 33452.09700638562, 'MAE_per_target': {'Total Alkalinity': 53.628780364990234, 'Electrical Conductance': 237.4567413330078, 'Dissolved Reactive Phosphorus': 40.614864349365234}, 'RMSE_per_target': {'Total Alkalinity': 69.48574829101562, 'Electrical Conductance': 304.8612365722656, 'Dissolved Reactive Phosphorus': 50.86886978149414}, 'R2_per_target': {'Total Alkalinity': -0.05453598499298096, 'Electrical Conductance': 0.032760679721832275, 'Dissolved Reactive Phosphorus': 0.11839514970779419}}
Trad
TEST: {'loss': 34121.611190382006, 'MAE_per_target': {'Total Alkalinity': 56.186527252197266, 'Electrical Conductance': 244.62864685058594, 'Dissolved Reactive Phosphorus': 38.42584991455078}, 'RMSE_per_target': {'Total Alkalinity': 69.4632797241211, 'Electrical Conductance': 308.0935363769531, 'Dissolved Reactive Phosphorus': 51.16688919067383}, 'R2_per_target': {'Total Alkalinity': -0.05385434627532959, 'Electrical Conductance': 0.012141644954681396, 'Dissolved Reactive Phosphorus': 0.10803496837615967}}
Lwir
TEST: {'loss': 33849.44345120366, 'MAE_per_target': {'Total Alkalinity': 54.50902557373047, 'Electrical Conductance': 240.64337158203125, 'Dissolved Reactive Phosphorus': 39.770503997802734}, 'RMSE_per_target': {'Total Alkalinity': 68.60441589355469, 'Electrical Conductance': 306.9317626953125, 'Dissolved Reactive Phosphorus': 51.3289794921875}, 'R2_per_target': {'Total Alkalinity': -0.027955174446105957, 'Electrical Conductance': 0.019577741622924805, 'Dissolved Reactive Phosphorus': 0.10237479209899902}}
Urad
TEST: {'loss': 31990.284795999094, 'MAE_per_target': {'Total Alkalinity': 51.60610580444336, 'Electrical Conductance': 228.6975555419922, 'Dissolved Reactive Phosphorus': 39.44300842285156}, 'RMSE_per_target': {'Total Alkalinity': 68.31578063964844, 'Electrical Conductance': 297.82171630859375, 'Dissolved Reactive Phosphorus': 51.049346923828125}, 'R2_per_target': {'Total Alkalinity': -0.019323348999023438, 'Electrical Conductance': 0.07691401243209839, 'Dissolved Reactive Phosphorus': 0.11212849617004395}}
Tcwi
TEST: {'loss': 32922.6649454679, 'MAE_per_target': {'Total Alkalinity': 53.02227020263672, 'Electrical Conductance': 237.251953125, 'Dissolved Reactive Phosphorus': 42.114749908447266}, 'RMSE_per_target': {'Total Alkalinity': 69.41325378417969, 'Electrical Conductance': 302.1748046875, 'Dissolved Reactive Phosphorus': 51.38286590576172}, 'R2_per_target': {'Total Alkalinity': -0.052336812019348145, 'Electrical Conductance': 0.04973244667053223, 'Dissolved Reactive Phosphorus': 0.10048913955688477}}
Evi
TEST: {'loss': 33341.17314294191, 'MAE_per_target': {'Total Alkalinity': 55.61763381958008, 'Electrical Conductance': 240.47926330566406, 'Dissolved Reactive Phosphorus': 38.54824447631836}, 'RMSE_per_target': {'Total Alkalinity': 68.90630340576172, 'Electrical Conductance': 304.4090576171875, 'Dissolved Reactive Phosphorus': 51.093685150146484}, 'R2_per_target': {'Total Alkalinity': -0.03702187538146973, 'Electrical Conductance': 0.03562784194946289, 'Dissolved Reactive Phosphorus': 0.11058551073074341}}
Osavi
TEST: {'loss': 33781.60524271022, 'MAE_per_target': {'Total Alkalinity': 54.88257598876953, 'Electrical Conductance': 241.47137451171875, 'Dissolved Reactive Phosphorus': 40.18063735961914}, 'RMSE_per_target': {'Total Alkalinity': 68.9951171875, 'Electrical Conductance': 306.5735168457031, 'Dissolved Reactive Phosphorus': 50.96242904663086}, 'R2_per_target': {'Total Alkalinity': -0.03969693183898926, 'Electrical Conductance': 0.02186506986618042, 'Dissolved Reactive Phosphorus': 0.11514943838119507}}
Gndvi
TEST: {'loss': 33863.081116495254, 'MAE_per_target': {'Total Alkalinity': 54.64398193359375, 'Electrical Conductance': 241.44049072265625, 'Dissolved Reactive Phosphorus': 39.45689010620117}, 'RMSE_per_target': {'Total Alkalinity': 69.58246612548828, 'Electrical Conductance': 306.8115539550781, 'Dissolved Reactive Phosphorus': 51.1291389465332}, 'R2_per_target': {'Total Alkalinity': -0.05747401714324951, 'Electrical Conductance': 0.020345568656921387, 'Dissolved Reactive Phosphorus': 0.10935068130493164}}
Gcvi
TEST: {'loss': 32971.48662127034, 'MAE_per_target': {'Total Alkalinity': 55.30144119262695, 'Electrical Conductance': 239.03515625, 'Dissolved Reactive Phosphorus': 39.74025344848633}, 'RMSE_per_target': {'Total Alkalinity': 69.2094497680664, 'Electrical Conductance': 302.5145568847656, 'Dissolved Reactive Phosphorus': 51.082794189453125}, 'R2_per_target': {'Total Alkalinity': -0.04616653919219971, 'Electrical Conductance': 0.04759418964385986, 'Dissolved Reactive Phosphorus': 0.11096465587615967}}
Msi
TEST: {'loss': 33288.04987002713, 'MAE_per_target': {'Total Alkalinity': 55.1786994934082, 'Electrical Conductance': 239.23098754882812, 'Dissolved Reactive Phosphorus': 40.06018829345703}, 'RMSE_per_target': {'Total Alkalinity': 69.68545532226562, 'Electrical Conductance': 304.04278564453125, 'Dissolved Reactive Phosphorus': 50.65639114379883}, 'R2_per_target': {'Total Alkalinity': -0.06060647964477539, 'Electrical Conductance': 0.037947237491607666, 'Dissolved Reactive Phosphorus': 0.12574470043182373}}
Bui
TEST: {'loss': 33595.02119476153, 'MAE_per_target': {'Total Alkalinity': 56.263484954833984, 'Electrical Conductance': 241.41554260253906, 'Dissolved Reactive Phosphorus': 39.89824295043945}, 'RMSE_per_target': {'Total Alkalinity': 69.52118682861328, 'Electrical Conductance': 305.4889221191406, 'Dissolved Reactive Phosphorus': 51.26778793334961}, 'R2_per_target': {'Total Alkalinity': -0.05561184883117676, 'Electrical Conductance': 0.028773725032806396, 'Dissolved Reactive Phosphorus': 0.10451376438140869}}
Tc Brightness
TEST: {'loss': 33650.80797567247, 'MAE_per_target': {'Total Alkalinity': 56.258140563964844, 'Electrical Conductance': 242.09231567382812, 'Dissolved Reactive Phosphorus': 38.844974517822266}, 'RMSE_per_target': {'Total Alkalinity': 69.48712921142578, 'Electrical Conductance': 305.7513122558594, 'Dissolved Reactive Phosphorus': 51.38166046142578}, 'R2_per_target': {'Total Alkalinity': -0.05457806587219238, 'Electrical Conductance': 0.027104437351226807, 'Dissolved Reactive Phosphorus': 0.10053133964538574}}
Tc Greenness
TEST: {'loss': 33791.47469767179, 'MAE_per_target': {'Total Alkalinity': 54.22063064575195, 'Electrical Conductance': 240.1296844482422, 'Dissolved Reactive Phosphorus': 39.107818603515625}, 'RMSE_per_target': {'Total Alkalinity': 68.90379333496094, 'Electrical Conductance': 306.5893859863281, 'Dissolved Reactive Phosphorus': 51.28001022338867}, 'R2_per_target': {'Total Alkalinity': -0.03694617748260498, 'Electrical Conductance': 0.021763861179351807, 'Dissolved Reactive Phosphorus': 0.10408681631088257}}
Nbr
TEST: {'loss': 33097.01940057075, 'MAE_per_target': {'Total Alkalinity': 55.86970901489258, 'Electrical Conductance': 239.90121459960938, 'Dissolved Reactive Phosphorus': 39.857261657714844}, 'RMSE_per_target': {'Total Alkalinity': 69.23097229003906, 'Electrical Conductance': 303.1153869628906, 'Dissolved Reactive Phosphorus': 51.17805862426758}, 'R2_per_target': {'Total Alkalinity': -0.04681730270385742, 'Electrical Conductance': 0.043807268142700195, 'Dissolved Reactive Phosphorus': 0.10764551162719727}}
Ndsi
TEST: {'loss': 31871.866784866637, 'MAE_per_target': {'Total Alkalinity': 52.26417541503906, 'Electrical Conductance': 229.17514038085938, 'Dissolved Reactive Phosphorus': 39.910560607910156}, 'RMSE_per_target': {'Total Alkalinity': 67.98789978027344, 'Electrical Conductance': 297.3558044433594, 'Dissolved Reactive Phosphorus': 50.7225227355957}, 'R2_per_target': {'Total Alkalinity': -0.00956273078918457, 'Electrical Conductance': 0.07979989051818848, 'Dissolved Reactive Phosphorus': 0.12346065044403076}}
Green/Red Ratio
TEST: {'loss': 34153.833020174054, 'MAE_per_target': {'Total Alkalinity': 57.110992431640625, 'Electrical Conductance': 246.65850830078125, 'Dissolved Reactive Phosphorus': 38.16859436035156}, 'RMSE_per_target': {'Total Alkalinity': 69.46406555175781, 'Electrical Conductance': 308.24127197265625, 'Dissolved Reactive Phosphorus': 51.220664978027344}, 'R2_per_target': {'Total Alkalinity': -0.053878188133239746, 'Electrical Conductance': 0.011193990707397461, 'Dissolved Reactive Phosphorus': 0.10615909099578857}}
Ndgi
TEST: {'loss': 33667.10161900995, 'MAE_per_target': {'Total Alkalinity': 55.70307159423828, 'Electrical Conductance': 243.739013671875, 'Dissolved Reactive Phosphorus': 38.61429214477539}, 'RMSE_per_target': {'Total Alkalinity': 69.30471801757812, 'Electrical Conductance': 305.9248352050781, 'Dissolved Reactive Phosphorus': 51.070167541503906}, 'R2_per_target': {'Total Alkalinity': -0.04904830455780029, 'Electrical Conductance': 0.026000022888183594, 'Dissolved Reactive Phosphorus': 0.11140406131744385}}
Ui (Urban Index)
TEST: {'loss': 34220.812245705245, 'MAE_per_target': {'Total Alkalinity': 55.329898834228516, 'Electrical Conductance': 242.05496215820312, 'Dissolved Reactive Phosphorus': 40.45228576660156}, 'RMSE_per_target': {'Total Alkalinity': 70.63058471679688, 'Electrical Conductance': 308.3127136230469, 'Dissolved Reactive Phosphorus': 51.156959533691406}, 'R2_per_target': {'Total Alkalinity': -0.08957123756408691, 'Electrical Conductance': 0.010735750198364258, 'Dissolved Reactive Phosphorus': 0.10838133096694946}}
Nbr2
TEST: {'loss': 34240.86851548372, 'MAE_per_target': {'Total Alkalinity': 55.97916030883789, 'Electrical Conductance': 243.96002197265625, 'Dissolved Reactive Phosphorus': 38.32744216918945}, 'RMSE_per_target': {'Total Alkalinity': 69.94795227050781, 'Electrical Conductance': 308.4877014160156, 'Dissolved Reactive Phosphorus': 51.62588119506836}, 'R2_per_target': {'Total Alkalinity': -0.0686119794845581, 'Electrical Conductance': 0.009612441062927246, 'Dissolved Reactive Phosphorus': 0.09196054935455322}}
Red/Nir Ratio
TEST: {'loss': 33933.4948999774, 'MAE_per_target': {'Total Alkalinity': 55.57752990722656, 'Electrical Conductance': 243.11328125, 'Dissolved Reactive Phosphorus': 39.86941146850586}, 'RMSE_per_target': {'Total Alkalinity': 70.08052825927734, 'Electrical Conductance': 307.07073974609375, 'Dissolved Reactive Phosphorus': 50.958351135253906}, 'R2_per_target': {'Total Alkalinity': -0.07266664505004883, 'Electrical Conductance': 0.018689632415771484, 'Dissolved Reactive Phosphorus': 0.11529088020324707}}
Green/Nir Ratio
TEST: {'loss': 33836.84357340642, 'MAE_per_target': {'Total Alkalinity': 53.57041931152344, 'Electrical Conductance': 239.5499267578125, 'Dissolved Reactive Phosphorus': 40.05455017089844}, 'RMSE_per_target': {'Total Alkalinity': 69.33411407470703, 'Electrical Conductance': 306.7832336425781, 'Dissolved Reactive Phosphorus': 50.8661003112793}, 'R2_per_target': {'Total Alkalinity': -0.04993867874145508, 'Electrical Conductance': 0.020526409149169922, 'Dissolved Reactive Phosphorus': 0.11849123239517212}}
Def
TEST: {'loss': 34248.832334990955, 'MAE_per_target': {'Total Alkalinity': 56.62791061401367, 'Electrical Conductance': 245.222900390625, 'Dissolved Reactive Phosphorus': 39.32936477661133}, 'RMSE_per_target': {'Total Alkalinity': 70.2198715209961, 'Electrical Conductance': 308.5123596191406, 'Dissolved Reactive Phosphorus': 51.33991622924805}, 'R2_per_target': {'Total Alkalinity': -0.07693648338317871, 'Electrical Conductance': 0.009454131126403809, 'Dissolved Reactive Phosphorus': 0.10199218988418579}}
Ppt_Roll3
TEST: {'loss': 32610.202058374773, 'MAE_per_target': {'Total Alkalinity': 52.21950912475586, 'Electrical Conductance': 233.62132263183594, 'Dissolved Reactive Phosphorus': 40.222171783447266}, 'RMSE_per_target': {'Total Alkalinity': 69.63179016113281, 'Electrical Conductance': 300.5040588378906, 'Dissolved Reactive Phosphorus': 51.76228332519531}, 'R2_per_target': {'Total Alkalinity': -0.05897367000579834, 'Electrical Conductance': 0.0602114200592041, 'Dissolved Reactive Phosphorus': 0.08715587854385376}}
Ppt_Roll6
TEST: {'loss': 34094.79429673372, 'MAE_per_target': {'Total Alkalinity': 52.590415954589844, 'Electrical Conductance': 239.0574188232422, 'Dissolved Reactive Phosphorus': 40.188697814941406}, 'RMSE_per_target': {'Total Alkalinity': 69.57242584228516, 'Electrical Conductance': 307.97369384765625, 'Dissolved Reactive Phosphorus': 50.95354080200195}, 'R2_per_target': {'Total Alkalinity': -0.05716884136199951, 'Electrical Conductance': 0.012910068035125732, 'Dissolved Reactive Phosphorus': 0.11545783281326294}}
Ppt_Roll12
TEST: {'loss': 35086.628609572785, 'MAE_per_target': {'Total Alkalinity': 56.84248733520508, 'Electrical Conductance': 246.69374084472656, 'Dissolved Reactive Phosphorus': 38.85221862792969}, 'RMSE_per_target': {'Total Alkalinity': 70.56283569335938, 'Electrical Conductance': 312.3649597167969, 'Dissolved Reactive Phosphorus': 52.04716110229492}, 'R2_per_target': {'Total Alkalinity': -0.08748197555541992, 'Electrical Conductance': -0.015439629554748535, 'Dissolved Reactive Phosphorus': 0.07708042860031128}}
Tmin
TEST: {'loss': 34184.02018817812, 'MAE_per_target': {'Total Alkalinity': 58.30690002441406, 'Electrical Conductance': 247.63528442382812, 'Dissolved Reactive Phosphorus': 38.69581604003906}, 'RMSE_per_target': {'Total Alkalinity': 69.33446502685547, 'Electrical Conductance': 308.1262512207031, 'Dissolved Reactive Phosphorus': 52.943458557128906}, 'R2_per_target': {'Total Alkalinity': -0.0499492883682251, 'Electrical Conductance': 0.011931896209716797, 'Dissolved Reactive Phosphorus': 0.04501974582672119}}
Et_Deficit
TEST: {'loss': 33850.28837731691, 'MAE_per_target': {'Total Alkalinity': 56.51723861694336, 'Electrical Conductance': 243.41586303710938, 'Dissolved Reactive Phosphorus': 39.02785110473633}, 'RMSE_per_target': {'Total Alkalinity': 69.72711181640625, 'Electrical Conductance': 306.6669616699219, 'Dissolved Reactive Phosphorus': 51.42345428466797}, 'R2_per_target': {'Total Alkalinity': -0.061874985694885254, 'Electrical Conductance': 0.021268784999847412, 'Dissolved Reactive Phosphorus': 0.09906744956970215}}
Pet_Ppt_Ratio
TEST: {'loss': 33155.169031843354, 'MAE_per_target': {'Total Alkalinity': 55.02865982055664, 'Electrical Conductance': 238.96490478515625, 'Dissolved Reactive Phosphorus': 38.931663513183594}, 'RMSE_per_target': {'Total Alkalinity': 68.75873565673828, 'Electrical Conductance': 303.53936767578125, 'Dissolved Reactive Phosphorus': 51.00587463378906}, 'R2_per_target': {'Total Alkalinity': -0.0325850248336792, 'Electrical Conductance': 0.04113048315048218, 'Dissolved Reactive Phosphorus': 0.11364001035690308}}
Runoff_Coefficient
TEST: {'loss': 33549.09961290687, 'MAE_per_target': {'Total Alkalinity': 54.206321716308594, 'Electrical Conductance': 239.48135375976562, 'Dissolved Reactive Phosphorus': 39.01920700073242}, 'RMSE_per_target': {'Total Alkalinity': 69.13385772705078, 'Electrical Conductance': 305.3752136230469, 'Dissolved Reactive Phosphorus': 51.12505340576172}, 'R2_per_target': {'Total Alkalinity': -0.04388248920440674, 'Electrical Conductance': 0.029496371746063232, 'Dissolved Reactive Phosphorus': 0.10949301719665527}}
Temp_Range
TEST: {'loss': 33539.8744525599, 'MAE_per_target': {'Total Alkalinity': 55.33556365966797, 'Electrical Conductance': 240.08465576171875, 'Dissolved Reactive Phosphorus': 38.66694641113281}, 'RMSE_per_target': {'Total Alkalinity': 69.70704650878906, 'Electrical Conductance': 305.1302185058594, 'Dissolved Reactive Phosphorus': 51.53739929199219}, 'R2_per_target': {'Total Alkalinity': -0.0612637996673584, 'Electrical Conductance': 0.031053245067596436, 'Dissolved Reactive Phosphorus': 0.095070481300354}}
Aet_Pet_Ration
TEST: {'loss': 33797.73299050633, 'MAE_per_target': {'Total Alkalinity': 53.81315612792969, 'Electrical Conductance': 239.3153533935547, 'Dissolved Reactive Phosphorus': 41.14042663574219}, 'RMSE_per_target': {'Total Alkalinity': 70.05156707763672, 'Electrical Conductance': 306.3974914550781, 'Dissolved Reactive Phosphorus': 51.05430603027344}, 'R2_per_target': {'Total Alkalinity': -0.07178008556365967, 'Electrical Conductance': 0.02298790216445923, 'Dissolved Reactive Phosphorus': 0.11195582151412964}}
Temp_Mean
TEST: {'loss': 34955.40398960217, 'MAE_per_target': {'Total Alkalinity': 56.40314483642578, 'Electrical Conductance': 250.3049774169922, 'Dissolved Reactive Phosphorus': 39.54596710205078}, 'RMSE_per_target': {'Total Alkalinity': 69.61997985839844, 'Electrical Conductance': 311.9777526855469, 'Dissolved Reactive Phosphorus': 51.857051849365234}, 'R2_per_target': {'Total Alkalinity': -0.05861461162567139, 'Electrical Conductance': -0.012923598289489746, 'Dissolved Reactive Phosphorus': 0.08381032943725586}}

"""

lines = raw_text.strip().splitlines()

results = {}
current_model = None

for line in lines:
    line = line.strip()
    if not line:
        continue

    # Model name line (e.g., "Emis")
    if not line.startswith("TEST:"):
        current_model = line
        results[current_model] = {}
        continue

    # TEST line
    if line.startswith("TEST:"):
        dict_str = line.replace("TEST:", "").strip()
        metrics = ast.literal_eval(dict_str)
        results[current_model] = metrics

# Convert to DataFrame
rows = []

for model_name, metrics in results.items():
    for target in metrics["MAE_per_target"].keys():
        rows.append({
            "Model": model_name,
            "Target": target,
            "MAE": metrics["MAE_per_target"][target],
            "RMSE": metrics["RMSE_per_target"][target],
            "R2": metrics["R2_per_target"][target],
            "Loss": metrics["loss"]
        })

df = pd.DataFrame(rows)
print(df)

              Model                         Target         MAE        RMSE  \
0              Emis               Total Alkalinity   56.311413   69.850792   
1              Emis         Electrical Conductance  244.264465  307.316223   
2              Emis  Dissolved Reactive Phosphorus   40.669659   51.648239   
3             Atran               Total Alkalinity   53.858631   68.799026   
4             Atran         Electrical Conductance  234.042984  299.974396   
..              ...                            ...         ...         ...   
124  Aet_Pet_Ration         Electrical Conductance  239.315353  306.397491   
125  Aet_Pet_Ration  Dissolved Reactive Phosphorus   41.140427   51.054306   
126       Temp_Mean               Total Alkalinity   56.403145   69.619980   
127       Temp_Mean         Electrical Conductance  250.304977  311.977753   
128       Temp_Mean  Dissolved Reactive Phosphorus   39.545967   51.857052   

           R2          Loss  
0   -0.065645  33996.644235  
1  

In [52]:
display(df)

df.to_csv('data/feature_comp.csv', index=False)

,Model,Target,MAE,RMSE,R2,Loss
0,Emis,Total Alkalinity,56.311413,69.850792,-0.065645,33996.644235
1,Emis,Electrical Conductance,244.264465,307.316223,0.017120,33996.644235
2,Emis,Dissolved Reactive Phosphorus,40.669659,51.648239,0.091174,33996.644235
3,Atran,Total Alkalinity,53.858631,68.799026,-0.033795,32463.353710
4,Atran,Electrical Conductance,234.042984,299.974396,0.063522,32463.353710
...,...,...,...,...,...,...
124,Aet_Pet_Ration,Electrical Conductance,239.315353,306.397491,0.022988,33797.732991
125,Aet_Pet_Ration,Dissolved Reactive Phosphorus,41.140427,51.054306,0.111956,33797.732991
126,Temp_Mean,Total Alkalinity,56.403145,69.619980,-0.058615,34955.403990
127,Temp_Mean,Electrical Conductance,250.304977,311.977753,-0.012924,34955.403990


In [ ]:
# session.sql("""
#     PUT file://output.csv
#     'snow://workspace/USER$.PUBLIC."EY-AI-and-Data-Challenge-version2"/versions/live/data/'
#     AUTO_COMPRESS=FALSE
#     OVERWRITE=TRUE
# """).collect()

# print("File saved! Refresh the browser to see the files in the sidebar")